# IIT Kharagpur Professor Discovery — Phase 2A Three-Department Pilot (`CS`, `ME`, `BT`)
## Native `BAAI/bge-m3` Dense (1024-d) + Learned Sparse (`lexical_weights`) GPU Embedding Notebook

**Instructions:**
1. Use a **T4 GPU** runtime in Google Colab or Kaggle.
2. Upload **only** `data/pilot/pilot_cs_me_bt_texts.jsonl` (contains only `content_hash` and `text_content` — no database credentials or PII).
3. Run all cells in order. Intermediate batches are checkpointed every 256 texts.
4. Download `pilot_cs_me_bt_embeddings_v1.jsonl.gz` and `pilot_embedding_manifest.json` and place them in `data/pilot/`.

In [ ]:
!pip install -q FlagEmbedding transformers torch

In [ ]:
import gzip
import json
import time
from pathlib import Path
import torch
import FlagEmbedding
import transformers
from FlagEmbedding import BGEM3FlagModel

INPUT_PATH = Path("pilot_cs_me_bt_texts.jsonl")
CHECKPOINT_PATH = Path("pilot_cs_me_bt_checkpoint.jsonl")
OUTPUT_GZ_PATH = Path("pilot_cs_me_bt_embeddings_v1.jsonl.gz")
MANIFEST_PATH = Path("pilot_embedding_manifest.json")

MODEL_NAME = "BAAI/bge-m3"
BATCH_SIZE = 32
MAX_LENGTH = 512
USE_FP16 = torch.cuda.is_available()

print(f"CUDA available: {torch.cuda.is_available()}, device: {torch.cuda.get_device_name(0) if torch.cuda.is_available() else 'CPU'}")
print(f"FlagEmbedding: {FlagEmbedding.__version__}, torch: {torch.__version__}, transformers: {transformers.__version__}")

In [ ]:
# Load exported unique texts
items = []
with open(INPUT_PATH, "r", encoding="utf-8") as f:
    for line in f:
        if line.strip():
            items.append(json.loads(line))
print(f"Loaded {len(items)} unique pilot texts.")

# Resume from checkpoint if present
done_hashes = set()
if CHECKPOINT_PATH.exists():
    with open(CHECKPOINT_PATH, "r", encoding="utf-8") as f:
        for line in f:
            if line.strip():
                done_hashes.add(json.loads(line)["content_hash"])
print(f"Already checkpointed: {len(done_hashes)}")

remaining = [x for x in items if x["content_hash"] not in done_hashes]
print(f"Remaining to encode: {len(remaining)}")

In [ ]:
if torch.cuda.is_available():
    torch.cuda.reset_peak_memory_stats()

t0 = time.time()
model = BGEM3FlagModel(MODEL_NAME, use_fp16=USE_FP16)

with open(CHECKPOINT_PATH, "a", encoding="utf-8") as ckpt_f:
    for start in range(0, len(remaining), BATCH_SIZE):
        batch = remaining[start : start + BATCH_SIZE]
        texts = [b["text_content"] for b in batch]
        out = model.encode(
            texts,
            batch_size=BATCH_SIZE,
            max_length=MAX_LENGTH,
            return_dense=True,
            return_sparse=True,
            return_colbert_vecs=False,
        )
        for rec, d_vec, lex_dict in zip(batch, out["dense_vecs"], out["lexical_weights"]):
            pairs = sorted(
                ((int(k), float(v)) for k, v in lex_dict.items() if float(v) > 0.0),
                key=lambda x: x[0],
            )
            row = {
                "content_hash": rec["content_hash"],
                "dense": [round(float(x), 6) for x in d_vec],
                "sparse_indices": [p[0] for p in pairs],
                "sparse_values": [round(p[1], 6) for p in pairs],
            }
            ckpt_f.write(json.dumps(row) + "\n")
        if (start // BATCH_SIZE) % 8 == 0:
            ckpt_f.flush()
            print(f"Encoded {min(start + BATCH_SIZE, len(remaining))}/{len(remaining)}")

elapsed = round(time.time() - t0, 2)
peak_gpu_mb = round(torch.cuda.max_memory_allocated() / (1024 ** 2), 2) if torch.cuda.is_available() else 0.0
print(f"Done in {elapsed}s | Peak GPU memory: {peak_gpu_mb} MB")

In [ ]:
# Compress final output & write manifest
count = 0
with open(CHECKPOINT_PATH, "r", encoding="utf-8") as src, gzip.open(OUTPUT_GZ_PATH, "wt", encoding="utf-8") as dst:
    for line in src:
        if line.strip():
            dst.write(line)
            count += 1

manifest = {
    "model_name": MODEL_NAME,
    "use_fp16": USE_FP16,
    "batch_size": BATCH_SIZE,
    "max_length": MAX_LENGTH,
    "vector_dim": 1024,
    "sparse_type": "bge_m3_lexical_weights",
    "unique_texts_encoded": count,
    "elapsed_seconds": elapsed,
    "peak_gpu_memory_mb": peak_gpu_mb,
    "library_versions": {
        "FlagEmbedding": FlagEmbedding.__version__,
        "torch": torch.__version__,
        "transformers": transformers.__version__,
    },
}
MANIFEST_PATH.write_text(json.dumps(manifest, indent=2), encoding="utf-8")
print(json.dumps(manifest, indent=2))